In [4]:
import duckdb
import pandas as pd

con = duckdb.connect()

BASE = "C:/Users/HP/Downloads/DS/da9/walmart-demand-forecast/data"

RAW = f"{BASE}/raw"

PROCESSED = f"{BASE}/processed"

# --- Check 1: duplicate key trước khi join ---
dup_check = con.execute(f"""
    SELECT Store, Dept, Date, COUNT(*) AS n_rows
    FROM read_csv_auto('{RAW}/train.csv')
    GROUP BY Store, Dept, Date
    HAVING COUNT(*) > 1
""").df()
print("=== DUPLICATE KEY CHECK ===")
print(f"Số dòng duplicate: {len(dup_check)}")
if len(dup_check) > 0:
    print(dup_check.head(10))

# --- Join chính ---
con.execute(f"""
    CREATE OR REPLACE TABLE walmart_joined AS
    SELECT
        t.Store, t.Dept, t.Date, t.Weekly_Sales, t.IsHoliday,
        f.Temperature, f.Fuel_Price,
        f.MarkDown1, f.MarkDown2, f.MarkDown3, f.MarkDown4, f.MarkDown5,
        f.CPI, f.Unemployment,
        s.Type AS Store_Type, s.Size AS Store_Size
    FROM read_csv_auto('{RAW}/train.csv') t
    LEFT JOIN read_csv_auto('{RAW}/features.csv') f
        ON t.Store = f.Store AND t.Date = f.Date
    LEFT JOIN read_csv_auto('{RAW}/stores.csv') s
        ON t.Store = s.Store
""")

# --- Check 2: số dòng có khớp sau join không  ---
n_raw = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{RAW}/train.csv')").fetchone()[0]
n_joined = con.execute("SELECT COUNT(*) FROM walmart_joined").fetchone()[0]
print("\n=== ROW COUNT CHECK SAU JOIN ===")
print(f"train.csv gốc: {n_raw:,} dòng")
print(f"Sau join     : {n_joined:,} dòng")
print(f"KHỚP: {n_raw == n_joined}" if n_raw == n_joined else "!!! LỆCH - có join sai key, DỪNG LẠI kiểm tra")

# --- Check 3: missing weeks theo Store-Dept ---
missing_weeks = con.execute("""
    WITH date_range AS (
        SELECT MIN(Date) AS min_d, MAX(Date) AS max_d FROM walmart_joined
    ),
    expected_weeks AS (
        SELECT COUNT(*) AS n_expected
        FROM generate_series(
            (SELECT min_d FROM date_range),
            (SELECT max_d FROM date_range),
            INTERVAL 7 DAY
        )
    )
    SELECT
        Store, Dept,
        COUNT(DISTINCT Date) AS n_actual_weeks,
        (SELECT n_expected FROM expected_weeks) AS n_expected_weeks,
        (SELECT n_expected FROM expected_weeks) - COUNT(DISTINCT Date) AS n_missing_weeks
    FROM walmart_joined
    GROUP BY Store, Dept
    HAVING n_missing_weeks > 0
    ORDER BY n_missing_weeks DESC
""").df()
print(f"\n=== MISSING WEEKS CHECK ===")
print(f"Số cặp Store-Dept bị thiếu tuần: {len(missing_weeks)}")
print(missing_weeks.head(15))

# --- Check 4: NA trong MarkDown - quyết định impute 0 hay giữ NA thật ---
markdown_na = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN MarkDown1 IS NULL THEN 1 ELSE 0 END) AS md1_null,
        SUM(CASE WHEN MarkDown2 IS NULL THEN 1 ELSE 0 END) AS md2_null,
        SUM(CASE WHEN MarkDown3 IS NULL THEN 1 ELSE 0 END) AS md3_null,
        SUM(CASE WHEN MarkDown4 IS NULL THEN 1 ELSE 0 END) AS md4_null,
        SUM(CASE WHEN MarkDown5 IS NULL THEN 1 ELSE 0 END) AS md5_null,
        MIN(Date) FILTER (WHERE MarkDown1 IS NOT NULL) AS first_md1_date
    FROM walmart_joined
""").df()
print(f"\n=== MARKDOWN NULL PATTERN ===")
print(markdown_na)


con.execute(f"""
    COPY walmart_joined TO '{PROCESSED}/walmart_joined.parquet' (FORMAT PARQUET)
""")
print(f"\n=== ĐÃ LƯU {PROCESSED}/walmart_joined.parquet ===")

=== DUPLICATE KEY CHECK ===
Số dòng duplicate: 0

=== ROW COUNT CHECK SAU JOIN ===
train.csv gốc: 421,570 dòng
Sau join     : 421,570 dòng
KHỚP: True

=== MISSING WEEKS CHECK ===
Số cặp Store-Dept bị thiếu tuần: 671
    Store  Dept  n_actual_weeks  n_expected_weeks  n_missing_weeks
0      36    29               1               143              142
1      15    99               1               143              142
2      15    37               1               143              142
3      21    96               1               143              142
4       3    78               1               143              142
5      16    99               1               143              142
6      36    71               1               143              142
7      33    71               1               143              142
8       7    99               1               143              142
9      18    48               1               143              142
10     25    96               1               1